# 2. Извлечение текстовых эмбеддингов

Берём предобученную модель `all-MiniLM-L6-v2` (sentence-transformers) и прогоняем через неё чанки субтитров.

Каждое видео → последовательность эмбеддингов (по одному на чанк) → вход для Transformer.

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import yaml
import warnings
warnings.filterwarnings("ignore")

with open("../configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

DATA_DIR = Path("../data")
EMB_MODEL = cfg["embedding"]["model_name"]
BATCH_SIZE = cfg["embedding"]["batch_size"]
MAX_SEQ_LEN = cfg["model"]["max_seq_len"]
EMB_DIM = cfg["embedding"]["embedding_dim"]

print(f"Embedding model: {EMB_MODEL}")
print(f"Embedding dim: {EMB_DIM}")
print(f"Max seq len: {MAX_SEQ_LEN}")

Embedding model: all-MiniLM-L6-v2
Embedding dim: 128
Max seq len: 15


In [9]:
model = SentenceTransformer(EMB_MODEL)
print(f"Model loaded: {EMB_MODEL}")
print(f"Max sequence length: {model.max_seq_length}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 354.92it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: all-MiniLM-L6-v2
Max sequence length: 256


## 2.1 Функция извлечения эмбеддингов

In [10]:
def extract_embeddings(df, model, batch_size=256, max_seq_len=20):
    """Извлекает эмбеддинги для каждого видео.
    
    Каждое видео имеет список чанков субтитров.
    Каждый чанк кодируется в вектор → видео = последовательность векторов.
    Обрезаем/паддим до max_seq_len.
    """
    import pandas as pd
    import numpy as np
    
    all_embeddings = []
    all_labels = []
    all_seq_lens = []
    
    # Собираем все чанки со всех видео в один список для batch-кодирования
    all_chunks = []
    video_indices = []  # (label, start, end, seq_len)
    
    for idx, row in df.iterrows():
        chunks = row["chunks"]
        
        # 🔧 Безопасная проверка на пустоту (работает с list, ndarray, None, NaN)
        if chunks.any() or pd.isna(chunks):
            chunks = []
        elif isinstance(chunks, np.ndarray):
            chunks = chunks.tolist()  # Конвертируем массив в список
        # Если chunks — список, оставляем как есть
        
        # Если после обработки список пустой — используем fallback
        if len(chunks) == 0:
            chunks = [row["full_text"][:512]]
        
        # Ограничиваем количество чанков
        chunks = chunks[:max_seq_len]
        
        start = len(all_chunks)
        all_chunks.extend(chunks)
        end = len(all_chunks)
        video_indices.append((row["label"], start, end, len(chunks)))
    
    print(f"Total chunks to encode: {len(all_chunks):,}")
    
    # Batch-кодирование всех чанков
    all_chunk_embeddings = model.encode(
        all_chunks, 
        batch_size=batch_size, 
        show_progress_bar=True,
        normalize_embeddings=True
    )
    
    # Собираем эмбеддинги обратно по видео
    emb_dim = all_chunk_embeddings.shape[1]
    
    for label, start, end, seq_len in video_indices:
        video_emb = np.zeros((max_seq_len, emb_dim), dtype=np.float32)
        video_emb[:seq_len] = all_chunk_embeddings[start:end]
        
        all_embeddings.append(video_emb)
        all_labels.append(label)
        all_seq_lens.append(seq_len)
    
    return np.array(all_embeddings), np.array(all_labels), np.array(all_seq_lens)
print("Function defined")

Function defined


## 2.2 Извлечение эмбеддингов для train / val / test

In [11]:
df_train = pd.read_parquet(DATA_DIR / "train.parquet")
df_val = pd.read_parquet(DATA_DIR / "val.parquet")
df_test = pd.read_parquet(DATA_DIR / "test.parquet")

print(f"Train: {len(df_train):,}, Val: {len(df_val):,}, Test: {len(df_test):,}")

Train: 137, Val: 30, Test: 30


In [12]:
print("=== Train ===")
train_emb, train_labels, train_lens = extract_embeddings(df_train, model, BATCH_SIZE, MAX_SEQ_LEN)
print(f"Shape: {train_emb.shape}")

print("\n=== Val ===")
val_emb, val_labels, val_lens = extract_embeddings(df_val, model, BATCH_SIZE, MAX_SEQ_LEN)
print(f"Shape: {val_emb.shape}")

print("\n=== Test ===")
test_emb, test_labels, test_lens = extract_embeddings(df_test, model, BATCH_SIZE, MAX_SEQ_LEN)
print(f"Shape: {test_emb.shape}")

=== Train ===
Total chunks to encode: 137


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.46it/s]


Shape: (137, 15, 384)

=== Val ===
Total chunks to encode: 30


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.76it/s]


Shape: (30, 15, 384)

=== Test ===
Total chunks to encode: 30


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.96it/s]

Shape: (30, 15, 384)


## 2.3 Сохранение эмбеддингов

In [13]:
np.savez_compressed(
    DATA_DIR / "embeddings.npz",
    train_emb=train_emb,
    train_labels=train_labels,
    train_lens=train_lens,
    val_emb=val_emb,
    val_labels=val_labels,
    val_lens=val_lens,
    test_emb=test_emb,
    test_labels=test_labels,
    test_lens=test_lens,
)

file_size = (DATA_DIR / "embeddings.npz").stat().st_size / (1024**2)
print(f"Saved embeddings.npz ({file_size:.1f} MB)")
print(f"Train: {train_emb.shape}, Val: {val_emb.shape}, Test: {test_emb.shape}")
print("\nEmbedding extraction complete! Upload embeddings.npz to Kaggle Dataset.")

Saved embeddings.npz (0.3 MB)
Train: (137, 15, 384), Val: (30, 15, 384), Test: (30, 15, 384)

Embedding extraction complete! Upload embeddings.npz to Kaggle Dataset.
